In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

import os
os.chdir('/Users/lior/DS6021/DS6021_F26/data')
penguins = pd.read_csv('penguins.csv')



In [2]:
#part a

cols = ['flipper_length_mm', 'bill_depth_mm', 'bill_length_mm', 'body_mass_g']
df = penguins.dropna(subset=cols).copy()
print(f"n = {len(df)} (rows common to all three models)")
 
def fit_model(predictors):
    X = sm.add_constant(df[predictors])
    y = df['body_mass_g']
    return sm.OLS(y, X).fit()
 
model_a = fit_model(['flipper_length_mm'])
model_b = fit_model(['flipper_length_mm', 'bill_depth_mm'])
model_c = fit_model(['flipper_length_mm', 'bill_depth_mm', 'bill_length_mm'])

n = 342 (rows common to all three models)


R2 is calculated from the total and residual sum of squares while adjusted subtracts out the number of predictors. Adding a predictor with R2 decreases the residual SS so it can never decrease, but Adj R2 can only increase when the predictor improves the model enough to the point where it can outweigh the penalties, so it can increase or decrease.

In [6]:
#part b

summary_table = pd.DataFrame({
    'Model': [
        'a.flipper_length',
        'b.flipper_length + bill_depth',
        'c.flipper_length + bill_depth + bill_length',
    ],
    'R2': [model_a.rsquared, model_b.rsquared, model_c.rsquared],
    'Adj R2':[model_a.rsquared_adj, model_b.rsquared_adj,model_c.rsquared_adj],
    'RSE': [np.sqrt(model_a.mse_resid), np.sqrt(model_b.mse_resid), np.sqrt(model_c.mse_resid)],
    'F-stat': [model_a.fvalue, model_b.fvalue, model_c.fvalue],
    'F p-value':[model_a.f_pvalue, model_b.f_pvalue, model_c.f_pvalue],
})


print("\nModel comparison:")
print(summary_table.round(4).to_string(index=False))


Model comparison:
                                      Model     R2  Adj R2      RSE    F-stat  F p-value
                           a.flipper_length 0.7590  0.7583 394.2782 1070.7446        0.0
              b.flipper_length + bill_depth 0.7610  0.7596 393.1784  539.8239        0.0
c.flipper_length + bill_depth + bill_length 0.7615  0.7594 393.4048  359.6718        0.0


4b

R2 is increasing with every added prediction while adjusted R2 is going all over where it increases and then decreases, meaning that adding bill length added nothing. The RSE is telling something similar but decreases from a to b then increases to c. I would choose the second model because it was the lowest RSE and the highest adj R2. 

In [3]:
#part c

print("\nModel b full summary:")
print(model_b.summary())
print("\nCoefficient on bill_depth_mm in model b:", model_b.params['bill_depth_mm'])


Model b full summary:
                            OLS Regression Results                            
Dep. Variable:            body_mass_g   R-squared:                       0.761
Model:                            OLS   Adj. R-squared:                  0.760
Method:                 Least Squares   F-statistic:                     539.8
Date:                Sat, 12 Sep 2026   Prob (F-statistic):          4.23e-106
Time:                        21:49:33   Log-Likelihood:                -2527.0
No. Observations:                 342   AIC:                             5060.
Df Residuals:                     339   BIC:                             5071.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const          

4c

The coefficient is 22.634 which is a positive regression since the flipper length is being held constant and its getting all the signals that were originally being led to the bill depth slope. Holding the flipper length at a fixed position and having a 1mm increase in bill depth and avg increase of 22.6g in body mass matches the true species relationship, so the multiple regression coeff is more accurately interprets this data.



In [4]:
print("\nModel c F-statistic:", model_c.fvalue, "p-value:", model_c.f_pvalue)


Model c F-statistic: 359.67180370851077 p-value: 8.188525644923907e-105


Null - no predictor has a linear relationship with bodt mass
Alt - at least one has a linear relationshop with body mass

F = 359.672 and p is almost equivalent to 0 so we reject the null hypothesis.